# Evaluate the GeoPlan Agent System

This notebook evaluates deterministic tool behaviour, live agent
tool selection, workflow routing, and final-report grounding.

Run it after the main workflow has produced its trace and report
files.


In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import json
import sys
import time

import pandas as pd
from IPython.display import display

from llama_index.llms.ollama import Ollama
from llama_index.core.agent.workflow import (
    ReActAgent,
    ToolCall,
    ToolCallResult,
)


## 1. Locate the project and load the deterministic tools


In [2]:
PROJECT_ROOT = Path.cwd().resolve()
SOURCE_DIR = PROJECT_ROOT / "src"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from geoplan_agent_tools import GeoPlanToolbox

toolbox = GeoPlanToolbox(
    PROJECT_ROOT,
    strict=True,
)

tools = toolbox.create_llamaindex_tools()


# Part A — Deterministic tool tests

These tests do not use an LLM.


In [3]:
selected_result = toolbox.get_selected_sequence(n=5)

if selected_result.get("status") != "ok":
    raise RuntimeError(
        "The selected sequence is unavailable."
    )

selected_ids = [
    record["candidate_id"]
    for record in selected_result["selection"]
]

candidate_a = selected_ids[0]
candidate_b = selected_ids[1]

print("Evaluation candidates:", candidate_a, candidate_b)


Evaluation candidates: green_p_710 green_p_821


In [4]:
deterministic_cases = []

def record_case(
    category: str,
    case_name: str,
    passed: bool,
    details: str,
) -> None:
    deterministic_cases.append(
        {
            "category": category,
            "case_name": case_name,
            "passed": bool(passed),
            "details": details,
        }
    )


result = toolbox.get_candidate(candidate_a)
record_case(
    "deterministic_tool",
    "valid_candidate_lookup",
    result.get("status") == "ok"
    and result.get("candidate", {}).get(
        "candidate_id"
    ) == candidate_a,
    str(result)[:500],
)


result = toolbox.get_candidate(
    "not_a_real_candidate"
)
record_case(
    "deterministic_tool",
    "unknown_candidate_handling",
    result.get("status") == "error"
    and result.get("error")
    == "unknown_candidate_id",
    str(result)[:500],
)


result = toolbox.compare_candidates(
    [candidate_a, candidate_b],
    metrics=[
        "overall_score",
        "demand_score",
        "coverage_score",
        "feasibility_score",
    ],
)
record_case(
    "deterministic_tool",
    "compare_two_candidates",
    result.get("status") == "ok"
    and result.get("candidate_ids")
    == [candidate_a, candidate_b],
    str(result)[:500],
)


result = toolbox.get_top_candidates(
    metric="overall_score",
    n=5,
    order="desc",
)
top_records = result.get("candidates", [])
top_scores = [
    record.get("overall_score")
    for record in top_records
]
descending = all(
    top_scores[index]
    >= top_scores[index + 1]
    for index in range(
        len(top_scores) - 1
    )
)
record_case(
    "deterministic_tool",
    "top_candidates_descending",
    result.get("status") == "ok"
    and len(top_records) == 5
    and descending,
    str(top_scores),
)


result = toolbox.audit_recommendation_set(
    candidate_ids=selected_ids
)
record_case(
    "deterministic_tool",
    "selected_set_audit",
    result.get("status") == "ok"
    and result.get("reviewer_approved")
    is True,
    str(result)[:800],
)


deterministic_results = pd.DataFrame(
    deterministic_cases
)
display(deterministic_results)


,category,case_name,passed,details
0,deterministic_tool,valid_candidate_lookup,True,"{'status': 'ok', 'candidate': {'candidate_id':..."
1,deterministic_tool,unknown_candidate_handling,True,"{'status': 'error', 'error': 'unknown_candidat..."
2,deterministic_tool,compare_two_candidates,True,"{'status': 'ok', 'candidate_ids': ['green_p_71..."
3,deterministic_tool,top_candidates_descending,True,"[0.7877772435, 0.7844610158, 0.7454211649, 0.7..."
4,deterministic_tool,selected_set_audit,True,"{'status': 'ok', 'candidate_count': 5, 'candid..."


# Part B — Saved multi-agent trace evaluation

Expected transition sequence:

```text
PlanningCoordinator
→ SiteEvidenceAgent
→ PlanningCoordinator
→ ScenarioRiskAgent
→ PlanningCoordinator
→ FinalReviewerAgent
→ PlanningCoordinator
```


In [5]:
TRACE_PATH = OUTPUT_DIR / "multiagent_trace.csv"
trace_cases = []

if TRACE_PATH.exists():
    trace = pd.read_csv(TRACE_PATH)
    trace_cases.append(
        {
            "category": "workflow_trace",
            "case_name": "trace_file_exists",
            "passed": True,
            "details": (
                f"{len(trace)} recorded events"
            ),
        }
    )
else:
    trace = pd.DataFrame()
    trace_cases.append(
        {
            "category": "workflow_trace",
            "case_name": "trace_file_exists",
            "passed": False,
            "details": str(TRACE_PATH),
        }
    )

display(trace.head(20))


,timestamp_utc,step,agent,event_type,tool_name,arguments,output_preview
0,2026-07-28T01:32:29.417076+00:00,1,PlanningCoordinator,AgentTransition,NaN,NaN,NaN
1,2026-07-28T01:32:50.118036+00:00,114,PlanningCoordinator,AgentOutput,handoff,NaN,```\nThought: The current language of the user...
2,2026-07-28T01:32:50.430860+00:00,115,PlanningCoordinator,ToolCall,handoff,"{""to_agent"": ""SiteEvidenceAgent"", ""reason"": ""T...",NaN
3,2026-07-28T01:32:50.431169+00:00,116,PlanningCoordinator,ToolCallResult,handoff,"{""to_agent"": ""SiteEvidenceAgent"", ""reason"": ""T...",Agent SiteEvidenceAgent is now handling the re...
4,2026-07-28T01:32:50.819231+00:00,117,SiteEvidenceAgent,AgentTransition,NaN,NaN,NaN
5,2026-07-28T01:33:10.029269+00:00,202,SiteEvidenceAgent,AgentOutput,get_selected_sequence,NaN,Thought: The current language of the user is: ...
6,2026-07-28T01:33:10.251369+00:00,203,SiteEvidenceAgent,ToolCall,get_selected_sequence,"{""n"": 5}",NaN
7,2026-07-28T01:33:10.261553+00:00,204,SiteEvidenceAgent,ToolCallResult,get_selected_sequence,"{""n"": 5}","{'status': 'ok', 'returned': 5, 'available': 2..."
8,2026-07-28T01:33:45.812569+00:00,460,SiteEvidenceAgent,AgentOutput,get_candidate,NaN,Thought: I now have the selected candidate sit...
9,2026-07-28T01:33:46.070253+00:00,461,SiteEvidenceAgent,ToolCall,get_candidate,"{""candidate_id"": ""green_p_710""}",NaN


In [6]:
def is_subsequence(
    expected: list[str],
    observed: list[str],
) -> bool:
    observed_index = 0

    for expected_item in expected:
        while (
            observed_index < len(observed)
            and observed[observed_index]
            != expected_item
        ):
            observed_index += 1

        if observed_index == len(observed):
            return False

        observed_index += 1

    return True


if not trace.empty:
    transitions = (
        trace.loc[
            trace["event_type"]
            == "AgentTransition",
            "agent",
        ]
        .dropna()
        .astype(str)
        .tolist()
    )

    expected_transitions = [
        "PlanningCoordinator",
        "SiteEvidenceAgent",
        "PlanningCoordinator",
        "ScenarioRiskAgent",
        "PlanningCoordinator",
        "FinalReviewerAgent",
        "PlanningCoordinator",
    ]

    trace_cases.append(
        {
            "category": "workflow_trace",
            "case_name": (
                "all_required_agents_participated"
            ),
            "passed": is_subsequence(
                expected_transitions,
                transitions,
            ),
            "details": (
                "Observed: "
                + " → ".join(transitions)
            ),
        }
    )

    called_tools = set(
        trace.loc[
            trace["event_type"] == "ToolCall",
            "tool_name",
        ]
        .dropna()
        .astype(str)
        .tolist()
    )

    required_tools = {
        "handoff",
        "get_selected_sequence",
        "get_candidate",
        "audit_recommendation_set",
    }

    trace_cases.append(
        {
            "category": "workflow_trace",
            "case_name": "required_tools_called",
            "passed": required_tools.issubset(
                called_tools
            ),
            "details": (
                "Called tools: "
                + ", ".join(
                    sorted(called_tools)
                )
            ),
        }
    )

    unknown_failures = trace[
        trace["output_preview"]
        .fillna("")
        .str.contains(
            "Tool .* not found",
            regex=True,
            case=False,
        )
    ]

    trace_cases.append(
        {
            "category": "workflow_trace",
            "case_name": (
                "no_unknown_tool_calls"
            ),
            "passed": unknown_failures.empty,
            "details": (
                f"{len(unknown_failures)} "
                "unknown-tool event(s)"
            ),
        }
    )


trace_results = pd.DataFrame(trace_cases)
display(trace_results)


,category,case_name,passed,details
0,workflow_trace,trace_file_exists,True,49 recorded events
1,workflow_trace,all_required_agents_participated,False,Observed: PlanningCoordinator → SiteEvidenceAg...
2,workflow_trace,required_tools_called,True,"Called tools: audit_recommendation_set, get_ca..."
3,workflow_trace,no_unknown_tool_calls,True,0 unknown-tool event(s)


# Part C — Saved final-report evaluation


In [7]:
def load_json_if_exists(
    path: Path,
) -> dict[str, Any] | None:
    if not path.exists():
        return None

    with path.open(
        "r",
        encoding="utf-8",
    ) as file:
        return json.load(file)


FINAL_REPORT_PATH = (
    OUTPUT_DIR / "final_agent_report.json"
)
VALIDATION_PATH = (
    OUTPUT_DIR
    / "final_agent_report_validation.json"
)
GROUNDING_PATH = (
    OUTPUT_DIR
    / "final_agent_report_grounding.json"
)

final_report_data = load_json_if_exists(
    FINAL_REPORT_PATH
)
validation_data = load_json_if_exists(
    VALIDATION_PATH
)
grounding_data = load_json_if_exists(
    GROUNDING_PATH
)

report_cases = []

for case_name, path, payload in [
    (
        "final_report_exists",
        FINAL_REPORT_PATH,
        final_report_data,
    ),
    (
        "validation_file_exists",
        VALIDATION_PATH,
        validation_data,
    ),
    (
        "grounding_file_exists",
        GROUNDING_PATH,
        grounding_data,
    ),
]:
    report_cases.append(
        {
            "category": "final_report",
            "case_name": case_name,
            "passed": payload is not None,
            "details": str(path),
        }
    )


if final_report_data is not None:
    sites = final_report_data.get(
        "recommended_sites",
        [],
    )
    ids = [
        site.get("candidate_id")
        for site in sites
    ]

    report_cases.append(
        {
            "category": "final_report",
            "case_name": (
                "candidate_ids_unique"
            ),
            "passed": (
                len(ids) == len(set(ids))
            ),
            "details": str(ids),
        }
    )

    report_cases.append(
        {
            "category": "final_report",
            "case_name": (
                "limitations_present"
            ),
            "passed": (
                len(
                    final_report_data.get(
                        "limitations",
                        [],
                    )
                )
                >= 3
            ),
            "details": str(
                final_report_data.get(
                    "limitations",
                    [],
                )
            )[:800],
        }
    )


if validation_data is not None:
    saved_checks = validation_data.get(
        "checks",
        {},
    )

    report_cases.append(
        {
            "category": "final_report",
            "case_name": (
                "saved_validation_passed"
            ),
            "passed": all(
                value is True
                for value in saved_checks.values()
                if isinstance(value, bool)
            ),
            "details": str(saved_checks),
        }
    )


if grounding_data is not None:
    report_cases.append(
        {
            "category": "final_report",
            "case_name": (
                "grounding_validation_passed"
            ),
            "passed": (
                grounding_data.get("grounded")
                is True
            ),
            "details": str(
                grounding_data.get(
                    "errors",
                    [],
                )
            ),
        }
    )


report_results = pd.DataFrame(
    report_cases
)
display(report_results)


,category,case_name,passed,details
0,final_report,final_report_exists,True,/Users/miladsaeedi/Desktop/Daily_Work_load/Pro...
1,final_report,validation_file_exists,True,/Users/miladsaeedi/Desktop/Daily_Work_load/Pro...
2,final_report,grounding_file_exists,True,/Users/miladsaeedi/Desktop/Daily_Work_load/Pro...
3,final_report,candidate_ids_unique,True,"['green_p_710', 'green_p_821', 'green_p_813', ..."
4,final_report,limitations_present,True,['The recommendations are planning-screening r...
5,final_report,saved_validation_passed,True,"{'pydantic_valid': True, 'grounding_valid': Tr..."
6,final_report,grounding_validation_passed,True,[]


# Part D — Live agent tool-selection tests

These focused tests check whether Qwen selects the expected deterministic tool.


In [8]:
MODEL_NAME = "qwen3:4b-instruct"

llm = Ollama(
    model=MODEL_NAME,
    base_url="http://localhost:11434",
    request_timeout=600.0,
    context_window=4096,
    temperature=0.0,
    keep_alive="0m",
)

print(llm.metadata)


context_window=4096 num_output=256 is_chat_model=True is_function_calling_model=True model_name='qwen3:4b-instruct' system_role=<MessageRole.SYSTEM: 'system'>


In [9]:
async def run_focused_case(
    *,
    case_name: str,
    prompt: str,
    allowed_tools: list[str],
    expected_tool: str,
    required_terms: list[str],
) -> dict[str, Any]:
    agent = ReActAgent(
        name=f"EvaluationAgent_{case_name}",
        description=(
            "Runs one focused GeoPlan evaluation."
        ),
        system_prompt=(
            "Use a GeoPlan tool before answering. "
            "Use only tool-returned facts. "
            "Do not invent candidate IDs or values. "
            "Answer concisely."
        ),
        tools=[
            tools[name]
            for name in allowed_tools
        ],
        llm=llm,
        streaming=True,
    )

    handler = agent.run(
        user_msg=prompt,
        max_iterations=8,
        early_stopping_method="generate",
    )

    called_tools = []
    start = time.perf_counter()

    async for event in handler.stream_events():
        if isinstance(event, ToolCall):
            called_tools.append(
                event.tool_name
            )

    response = await handler
    elapsed = time.perf_counter() - start
    response_text = str(response)

    expected_tool_called = (
        expected_tool in called_tools
    )
    required_terms_present = all(
        term.lower() in response_text.lower()
        for term in required_terms
    )

    return {
        "category": "live_agent",
        "case_name": case_name,
        "passed": (
            expected_tool_called
            and required_terms_present
        ),
        "details": response_text[:1000],
        "expected_tool": expected_tool,
        "called_tools": ", ".join(
            called_tools
        ),
        "latency_seconds": round(
            elapsed,
            2,
        ),
    }


In [10]:
live_cases = []

live_cases.append(
    await run_focused_case(
        case_name="candidate_lookup",
        prompt=(
            f"Retrieve evidence for {candidate_a}. "
            "State its exact candidate ID and address."
        ),
        allowed_tools=["get_candidate"],
        expected_tool="get_candidate",
        required_terms=[candidate_a],
    )
)

live_cases.append(
    await run_focused_case(
        case_name="candidate_comparison",
        prompt=(
            f"Compare {candidate_a} and {candidate_b} "
            "using overall_score, demand_score, "
            "coverage_score, feasibility_score, "
            "and equity_score."
        ),
        allowed_tools=[
            "compare_candidates",
            "get_candidate",
        ],
        expected_tool="compare_candidates",
        required_terms=[
            candidate_a,
            candidate_b,
        ],
    )
)

live_cases.append(
    await run_focused_case(
        case_name="selected_sequence",
        prompt=(
            "Return the first three deterministic "
            "selected sites in order."
        ),
        allowed_tools=[
            "get_selected_sequence"
        ],
        expected_tool=(
            "get_selected_sequence"
        ),
        required_terms=[
            selected_ids[0],
            selected_ids[1],
            selected_ids[2],
        ],
    )
)

live_results = pd.DataFrame(
    live_cases
)
display(live_results)


,category,case_name,passed,details,expected_tool,called_tools,latency_seconds
0,live_agent,candidate_lookup,True,The exact candidate ID is green_p_710 and its ...,get_candidate,get_candidate,25.40
1,live_agent,candidate_comparison,True,"green_p_710 has a higher overall_score, demand...",compare_candidates,compare_candidates,25.53
2,live_agent,selected_sequence,False,The first three deterministic selected sites i...,get_selected_sequence,get_selected_sequence,34.89


# Part E — Consolidated score and export


In [11]:
all_results = pd.concat(
    [
        deterministic_results,
        trace_results,
        report_results,
        live_results[
            [
                "category",
                "case_name",
                "passed",
                "details",
            ]
        ],
    ],
    ignore_index=True,
)

all_results["passed"] = (
    all_results["passed"].fillna(False)
)

summary = (
    all_results
    .groupby(
        "category",
        as_index=False,
    )
    .agg(
        tests=("case_name", "count"),
        passed=("passed", "sum"),
    )
)

summary["pass_rate"] = (
    summary["passed"]
    / summary["tests"]
)

overall_pass_rate = float(
    all_results["passed"].mean()
)

display(all_results)
display(summary)

print(
    "Overall pass rate:",
    f"{overall_pass_rate:.1%}",
)


,category,case_name,passed,details
0,deterministic_tool,valid_candidate_lookup,True,"{'status': 'ok', 'candidate': {'candidate_id':..."
1,deterministic_tool,unknown_candidate_handling,True,"{'status': 'error', 'error': 'unknown_candidat..."
2,deterministic_tool,compare_two_candidates,True,"{'status': 'ok', 'candidate_ids': ['green_p_71..."
3,deterministic_tool,top_candidates_descending,True,"[0.7877772435, 0.7844610158, 0.7454211649, 0.7..."
4,deterministic_tool,selected_set_audit,True,"{'status': 'ok', 'candidate_count': 5, 'candid..."
5,workflow_trace,trace_file_exists,True,49 recorded events
6,workflow_trace,all_required_agents_participated,False,Observed: PlanningCoordinator → SiteEvidenceAg...
7,workflow_trace,required_tools_called,True,"Called tools: audit_recommendation_set, get_ca..."
8,workflow_trace,no_unknown_tool_calls,True,0 unknown-tool event(s)
9,final_report,final_report_exists,True,/Users/miladsaeedi/Desktop/Daily_Work_load/Pro...


,category,tests,passed,pass_rate
0,deterministic_tool,5,5,1.000000
1,final_report,7,7,1.000000
2,live_agent,3,2,0.666667
3,workflow_trace,4,3,0.750000


Overall pass rate: 89.5%


In [12]:
CASES_PATH = (
    OUTPUT_DIR
    / "agent_evaluation_cases.csv"
)
SUMMARY_CSV_PATH = (
    OUTPUT_DIR
    / "agent_evaluation_summary.csv"
)
SUMMARY_JSON_PATH = (
    OUTPUT_DIR
    / "agent_evaluation_summary.json"
)

all_results.to_csv(
    CASES_PATH,
    index=False,
)
summary.to_csv(
    SUMMARY_CSV_PATH,
    index=False,
)

payload = {
    "evaluated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "model_name": MODEL_NAME,
    "overall_pass_rate": (
        overall_pass_rate
    ),
    "categories": summary.to_dict(
        orient="records"
    ),
    "failed_cases": (
        all_results.loc[
            ~all_results["passed"],
            [
                "category",
                "case_name",
                "details",
            ],
        ]
        .to_dict(orient="records")
    ),
}

with SUMMARY_JSON_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        payload,
        file,
        indent=2,
        default=str,
    )

print("Saved:", CASES_PATH.resolve())
print("Saved:", SUMMARY_CSV_PATH.resolve())
print("Saved:", SUMMARY_JSON_PATH.resolve())


Saved: /Users/miladsaeedi/Desktop/Daily_Work_load/Projects_for_CV/Geospatial_site_selection/outputs/agent_evaluation_cases.csv
Saved: /Users/miladsaeedi/Desktop/Daily_Work_load/Projects_for_CV/Geospatial_site_selection/outputs/agent_evaluation_summary.csv
Saved: /Users/miladsaeedi/Desktop/Daily_Work_load/Projects_for_CV/Geospatial_site_selection/outputs/agent_evaluation_summary.json


# Portfolio acceptance criteria

A strong prototype should show:

- deterministic tool tests passing;
- focused agents choosing the expected tools;
- no unknown tool calls;
- all four workflow agents appearing in the trace;
- final report validation passing;
- deterministic grounding validation passing.

A failed routing test should be fixed and rerun rather than hidden.
